# Farsi floret vectors on Colab (Weasel pipeline)

Persian Wikipedia -> floret vectors on GPU -> an installable spaCy model you can `spacy.load("fa_floret")` later.

**Runtime -> Change runtime type -> T4 GPU** first.

Adapted from explosion/projects `floret_wiki_oscar_vectors`. Three things had to change to run here: OSCAR is gated now, `datasets>=3` removed `use_auth_token=`, and the upstream defaults assume a 16-core box with `/scratch`.

In [ ]:
# 1 -- GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch, os; print('cuda', torch.cuda.is_available(), '| vCPUs', os.cpu_count())

## Clone the private repo
Put a GitHub PAT (scope `repo`) in Colab **Secrets** (key icon, left sidebar) named `GH_TOKEN`, with notebook access enabled. Nothing is printed or written to disk.

In [ ]:
# 2 -- clone
from google.colab import userdata
import subprocess
tok = userdata.get('GH_TOKEN')
url = f'https://{tok}@github.com/Fazel94/floret-torch.git'
subprocess.run(['git','clone','-q',url,'/content/floret-torch'], check=True)
%cd /content/floret-torch
!git log --oneline -1

In [ ]:
# 3 -- dependencies. `-e .` puts floret_torch on the path so the weasel
#      commands can run from the project dir. torch is already present.
!pip -q install -e . 2>&1 | tail -2
!pip -q install spacy weasel datasets floret mmh3 2>&1 | tail -2
import spacy, weasel, floret_torch; print('spacy', spacy.__version__, '| ok')

In [ ]:
# 4 -- parity gate: our hashing must match CPU floret byte for byte
!python -m pytest tests/test_parity.py -q

## Build the corpus
`max_texts` is the knob. Persian Wikipedia is ~1M articles; tokenizing measures ~1.08M tokens/s on one core, so CPU is not the bottleneck — the HF stream download is.

| max_texts | tokens | rough wall time |
|---|---|---|
| 50,000 | ~20M | few min |
| 100,000 | ~41M | ~10 min |
| 400,000 | ~163M | ~30 min |

100k is a good first run; 400k gives noticeably better vectors.

In [ ]:
# 5 -- stream Persian Wikipedia -> one tokenized sentence per line
%cd /content/floret-torch/colab/farsi_weasel
!python -m weasel run tokenize-wikipedia . --vars.max_texts 100000
!wc -l /content/corpus/fa.txt && head -2 /content/corpus/fa.txt

In [ ]:
# 6 -- train on GPU. Keep --batch in 4096-16384: the input-row gradient is
#      un-normalized (as fastText does it), so an oversized batch accumulates
#      hundreds of full gradients onto one row and diverges.
!python -m weasel run train-vectors-gpu . --vars.vector_epoch 5 --vars.vector_batch 8192

In [ ]:
# 7 -- wrap into a spaCy pipeline and build an installable wheel
!python -m weasel run init-vectors .
!python -m weasel run package-vectors .
!find /content/packages -name '*.whl'

## Persist it — Colab wipes `/content` when the runtime recycles
This copies the `.floret` table, the `.vec` file, the zipped pipeline and the wheel to Drive. The `.floret` table is the portable artifact: it rebuilds the pipeline anywhere via `spacy init vectors --mode floret`.

In [ ]:
# 8 -- save to Drive (mounts it if needed)
!python -m weasel run save-to-drive .

In [ ]:
# 9 -- install and use, exactly as you would any spaCy model
import glob
whl = glob.glob('/content/packages/**/*.whl', recursive=True)[0]
!pip -q install --force-reinstall {whl}

import spacy
nlp = spacy.load('fa_floret')
print(nlp.vocab.vectors.shape, nlp.vocab.vectors.mode)

doc = nlp('کتاب‌های خوبی در دانشگاه تهران نوشتند')
for t in doc:
    print(f'{t.text:<14} norm={t.vector_norm:.3f} has_vector={t.has_vector}')

print('sim:', nlp('دانشگاه').similarity(nlp('دانشکده')))
# floret has no fixed vocab: a never-seen word still gets a real vector
print('unseen:', nlp('واژه‌ای‌که‌هرگزدیده‌نشده')[0].has_vector)

In [ ]:
# 10 -- neighbour sanity check
!python -m weasel run evaluate-vectors .

## Using it later, in a fresh session
Nothing above needs re-running. Either install the wheel from Drive, or rebuild from the `.floret` table.

In [ ]:
# LATER -- option A: install the wheel straight from Drive
from google.colab import drive; drive.mount('/content/drive')
!pip -q install /content/drive/MyDrive/floret_fa/fa_floret-0.1.0-py3-none-any.whl
import spacy; nlp = spacy.load('fa_floret'); print(nlp.vocab.vectors.shape)

In [ ]:
# LATER -- option B: rebuild the pipeline from the portable .floret table
!python -m spacy init vectors fa \
  /content/drive/MyDrive/floret_fa/fa.floret /content/fa_pipeline \
  --mode floret --name floret
import spacy; nlp = spacy.load('/content/fa_pipeline'); print(nlp.vocab.vectors.shape)

## Using the vectors to train a real component
Vectors are an input to training, not a model. To train a Persian tagger/parser on top of them, point the config's `[initialize] vectors` at the pipeline directory and set `[components.tok2vec] include_static_vectors = true`:

```bash
python -m spacy init config config.cfg --lang fa --pipeline tagger,parser --optimize accuracy
python -m spacy train config.cfg \
  --paths.train train.spacy --paths.dev dev.spacy \
  --initialize.vectors /content/vectors/fa_floret \
  --components.tok2vec.model.embed.include_static_vectors true \
  --gpu-id 0
```

Persian treebanks for `train.spacy`: `UD_Persian-Seraji` or `UD_Persian-PerDT`, converted with `spacy convert file.conllu ./ --converter conllu`.